# [Programming Z3](https://theory.stanford.edu/~nikolaj/programmingz3.html?utm_source=chatgpt.com#sec-logical-interfaces)

## 2. Logical Interfaces to Z3

In [1]:
from z3 import Bool, BoolRef, Not, Or, Solver, sat

tie: BoolRef = Bool("Tie")
shirt: BoolRef = Bool("Shirt")

s: Solver = Solver()
s.add(
    Or(tie, shirt),
    Or(Not(tie), shirt),
    Or(Not(tie), Not(shirt)),
)

result = s.check()
print(result)

if result == sat:
    print(s.model())

sat
[Tie = False, Shirt = True]


In [2]:
from z3 import (
    Array,
    ArrayRef,
    ArithRef,
    BoolRef,
    FuncDeclRef,
    Function,
    Implies,
    Int,
    IntSort,
    Not,
    Solver,
    SortRef,
    Store,
)

Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", Z, Z)  # 非解釈関数

x: ArithRef = Int("x")
y: ArithRef = Int("y")
z: ArithRef = Int("z")

A: ArrayRef = Array("A", Z, Z)

fml: BoolRef = Implies(
    x + 2 == y,
    # Store(A, x, 3) は、配列 A の添字 x に値 3 を格納した新しい配列を表す
    f(Store(A, x, 3)[y - 2]) == f(y - x + 1),
)

s: Solver = Solver()
s.add(Not(fml))  # 否定

print(s.check())  # unsat

unsat


In [3]:
from z3 import ArithRef, Int, Solver

x: ArithRef = Int("x")
y: ArithRef = Int("y")

s: Solver = Solver()

s.add((x % 4) + 3 * (y / 2) > x - y)

print(s.sexpr())

(declare-fun y () Int)
(declare-fun x () Int)
(assert (> (+ (mod x 4) (* 3 (div y 2))) (- x y)))



In [4]:
from z3 import Bool, BoolRef, Implies, Not, Solver

a: BoolRef = Bool("a")
b: BoolRef = Bool("b")
r: BoolRef = Bool("r")

s: Solver = Solver()
s.add(
    Implies(a, r),
    Implies(b, r),
    a,
)

# 現在の公理集合から r が導けるか？
s.push()
s.add(Not(r))

print(s.check())  # unsat

s.pop()

unsat


### 2.1. Sorts

$$
S^{\mathcal M}\neq\varnothing
$$

となる議論領域を持つ Sort \(S\) を宣言したことに

In [5]:
from z3 import (
    BoolRef,
    Const,
    DeclareSort,
    ExprRef,
    ForAll,
    Solver,
    SortRef,
)

S: SortRef = DeclareSort("S")  # 新しい Sort を定義
s: ExprRef = Const("s", S)  # Sort S に属する定数記号 s の宣言

solver: Solver = Solver()
solver.add(ForAll([s], s != s))

print(solver.check())  # unsat

unsat


### 2.2 Signatures


$$
f:\mathbb Z\to\mathbb Z
$$

$$
P:\mathbb Z\to\mathbb B
$$

Z3 では Predicate も Boolean-valued Function として宣言

In [6]:
from z3 import (
    BoolSort,
    ExprRef,
    FuncDeclRef,
    Function,
    IntSort,
    SortRef,
)

Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", Z, Z)  # 関数記号
P: FuncDeclRef = Function("P", Z, BoolSort())  # 述語記号

term = f(x)  # 項

### 2.3 Terms and Formulas

In [7]:
from z3 import (
    Bool,
    BoolRef,
    BoolSort,
    FuncDeclRef,
    Function,
    IntSort,
    Solver,
    SortRef,
)

B: SortRef = BoolSort()
Z: SortRef = IntSort()

f: FuncDeclRef = Function("f", B, Z)
g: FuncDeclRef = Function("g", Z, B)

a: BoolRef = Bool("a")  # 定数記号

solver: Solver = Solver()
solver.add(g(1 + f(a)))

print(solver.check())  # sat
print(solver.model())

sat
[a = False, g = [else -> True], f = [else -> 0]]


In [8]:
from z3 import ArithRef, BoolRef, Int

x: ArithRef = Int("x")
y: ArithRef = Int("y")

n: BoolRef = x + y >= 3

print("num args:", n.num_args())
print("children:", n.children())
print("1st child:", n.arg(0))
print("2nd child:", n.arg(1))
print("operator:", n.decl())
print("op name:", n.decl().name())

num args: 2
children: [x + y, 3]
1st child: x + y
2nd child: 3
operator: >=
op name: >=


### 2.4. Quantifiers and Lambda binding

$$
\underbrace{y=x+1}_{\text{外側の制約}}
\quad\land\quad
\underbrace{\forall y\;(y\le 0\to x<y)}_{\text{量化式}}
$$

ここで、外側の $y$ と量化式の内側の $y$ は、別の役割

In [9]:
from z3 import ArithRef, BoolRef, ForAll, Implies, Int, Solver

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# 例1
phi1: BoolRef = ForAll(
    [y],
    Implies(y <= 0, x < y),
)

s1: Solver = Solver()
s1.add(y == x + 1, phi1)
print(s1.check())  # unsat


unsat


In [10]:
# 例2
phi2: BoolRef = ForAll(
    [y],
    Implies(y <= 0, x > y),
)

s2: Solver = Solver()
s2.add(y == x + 1, phi2)
print(s2.check())  # sat
print(s2.model())

sat
[x = 2, y = 3]


#### lambda

$$
m'=
\lambda x.
\begin{cases}
v & l\le x\le h\\
m(x) & \text{otherwise}
\end{cases}
$$

ここで
- $x$：Lambda が束縛する変数
- $l,h$：更新範囲
- $v$：設定したい値
- $m$：元の配列
- $m'$：新しく構築した配列

In [11]:
from z3 import (
    And,
    ArithRef,
    Array,
    ArrayRef,
    If,
    Int,
    IntSort,
    Lambda,
    QuantifierRef,
    Select,
    Solver,
)

m: ArrayRef = Array("m", IntSort(), IntSort())
m1: ArrayRef = Array("m1", IntSort(), IntSort())
z: ArithRef = Int("z")


def memset(
    lo: int,
    hi: int,
    value: ArithRef,
    base: ArrayRef,
) -> QuantifierRef:
    x: ArithRef = Int("x")
    
    # Lambda：変数を束縛して関数を構築する
    return Lambda(
        [x],
        If(
            And(lo <= x, x <= hi),
            value,
            Select(base, x),
        ),
    )


updated: QuantifierRef = memset(1, 700, z, m)

s: Solver = Solver()
s.add(
    m1 == updated,
    Select(m1, 6) != z,
)
print(s.check())  # unsat

unsat


#### 述語を量化

In [12]:
from z3 import (
    ArithRef,
    Array,
    ArrayRef,
    BoolRef,
    BoolSort,
    ForAll,
    Implies,
    Int,
    IntSort,
    Not,
    Select,
    Solver,
)

x: ArithRef = Int("x")
y: ArithRef = Int("y")

# 一部の二階論理的な推論を表現できる
Q: ArrayRef = Array("Q", IntSort(), BoolSort())

phi: BoolRef = Implies(
    ForAll(
        [Q],
        Implies(Select(Q, x), Select(Q, y)),
    ),
    x == y,
)

s: Solver = Solver()
s.add(Not(phi))

print(s.check())  # unsat


unsat


## 3. Theories

### 3.1. EUF: Equality and Uninterpreted Functions

EUF は Equality and Uninterpreted Functions の略

$$
a=b\implies f(a)=f(b)
$$

$$
a\ne b\implies f(a)\ne f(b)
$$

---

以下サンプルコード

$$
\begin{aligned}
(1)\quad & f(f(x))=x\\
(2)\quad & f(f(f(x)))=x
\end{aligned}
$$

(1) の両辺に \(f\) を適用
$$
f(f(f(x)))=f(x)
$$
したがって
$$
\boxed{f(x)=x}
$$

In [13]:
from z3 import (
    Const,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
)

S: SortRef = DeclareSort("S")
f: FuncDeclRef = Function("f", S, S)
x: ExprRef = Const("x", S)

solver: Solver = Solver()

solver.add(
    f(f(x)) == x,
    f(f(f(x))) == x,
)

print(solver.check())  # sat

solver.add(f(x) != x)

print(solver.check())  # unsat

sat
unsat


##### 同値類，union-find

等式の推移律  
when
$$
a=b,\qquad b=c
$$

then
$$
a=c
$$

In [14]:
from z3 import Consts, DeclareSort, ExprRef, Solver, SortRef

S: SortRef = DeclareSort("S")
a, b, c = Consts("a b c", S)

solver: Solver = Solver()
solver.add(a == b, b == c)

print(solver.check())  # sat

solver.add(a != c)
print(solver.check())  # unsat

sat
unsat


合同律（Congruence Rule）
$$
\boxed{
\bigwedge_{i=1}^{n}(t_i=u_i)
\implies
f(t_1,\ldots,t_n)=f(u_1,\ldots,u_n)
}
$$

In [15]:
from z3 import (
    Consts,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
)

S: SortRef = DeclareSort("S")
a, b, d, e = Consts("a b d e", S)

g: FuncDeclRef = Function("g", S, S)
f: FuncDeclRef = Function("f", S, S, S)

solver: Solver = Solver()
solver.add(
    a == b,
    d == e,
    b == g(d),
    f(a, g(d)) != f(g(e), b),
)

print(solver.check())  # unsat

unsat


#### 3.1.1. Congruence Closure

- $\mathcal T$：項の集合（set of terms）  
- $\mathcal E$：$\mathcal T$ 上の等式の集合  
- $cc$：Congruence Closure。まず $\mathcal T$ の分割として導入  
- $s,t$：$\mathcal T$ に属する項  
- $s_i,t_i$：関数適用の引数となる項  

$$
(s=t)\in\mathcal E
\quad\Longrightarrow\quad
s,t\text{ are in the same partition in }cc.
$$

---

$$
s:=f(s_1,\ldots,s_k),\qquad
t:=f(t_1,\ldots,t_k)
$$

という2つの項 $s,t\in\mathcal T$ について、
$$
\boxed{
\left(
\forall i=1,\ldots,k,\;
cc(s_i)=cc(t_i)
\right)
\Longrightarrow
cc(s)=cc(t)
}
$$

---

$$
\boxed{
cc:\mathcal T\to 2^{\mathcal T}
}
$$
ここで，
$$
2^{\mathcal T}
=
\{X\mid X\subseteq\mathcal T\}
$$

In [16]:
from z3 import (
    Const,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
)

S: SortRef = DeclareSort("S")

a: ExprRef = Const("a", S)
b: ExprRef = Const("b", S)
c: ExprRef = Const("c", S)

f: FuncDeclRef = Function("f", S, S)
g: FuncDeclRef = Function("g", S, S)

solver: Solver = Solver()
solver.add(
    a == b,
    f(a) == c,
)

# 合同性から f(b) == c が導かれるか？
solver.push()
solver.add(f(b) != c)
print("1:", solver.check())  # unsat
solver.pop()

# 関数をさらに適用した場合も成立するか？
solver.push()
solver.add(g(f(a)) != g(f(b)))
print("2:", solver.check())  # unsat
solver.pop()

# f(c) == f(a) まで導かれるか？
solver.push()
solver.add(f(c) != f(a))
print("3:", solver.check())  # sat
solver.pop()

1: unsat
2: unsat
3: sat


#### 3.1.2. EUF Models

EUF のモデル構成: CCによる同値類に**具体的な値**を割り当て、関数記号の解釈を構築する

---

例：

$$
f:S\times S\to S,\qquad g:S\to S
$$
$$
\boxed{
\begin{aligned}
&a=b\\
{}\land{}&b=c\\
{}\land{}&d=e\\
{}\land{}&b=s\\
{}\land{}&d=t\\
{}\land{}&f(a,g(d))\ne f(g(e),b)
\end{aligned}
}
$$

S!val!0 のような名前：Z3 がモデルの中で用意した、Sort $S$ の具体的な要素
$$
\boxed{
S^{\mathcal M}
=
\{
\texttt{S!val!0},
\texttt{S!val!1},
\texttt{S!val!2},
\texttt{S!val!3},
\texttt{S!val!4}
\}
}
$$
$$
g(d)=g(e)=\texttt{S!val!2}
$$
$$
f^{\mathcal M}(x,y)=
\begin{cases}
\texttt{S!val!4}
& (x,y)=(\texttt{S!val!2},\texttt{S!val!0})\\
\texttt{S!val!3}
& \text{それ以外}
\end{cases}
$$

In [17]:
from z3 import (
    Const,
    DeclareSort,
    ExprRef,
    FuncDeclRef,
    Function,
    Solver,
    SortRef,
    sat,
)

S: SortRef = DeclareSort("S")

a: ExprRef = Const("a", S)
b: ExprRef = Const("b", S)
c: ExprRef = Const("c", S)
d: ExprRef = Const("d", S)
e: ExprRef = Const("e", S)
s: ExprRef = Const("s", S)
t: ExprRef = Const("t", S)

f: FuncDeclRef = Function("f", S, S, S)
g: FuncDeclRef = Function("g", S, S)

solver: Solver = Solver()
solver.add(
    a == b,
    b == c,
    d == e,
    b == s,
    d == t,
    f(a, g(d)) != f(g(e), b),
)

if solver.check() == sat:
    print(solver.model())

[c = S!val!0,
 e = S!val!1,
 a = S!val!0,
 s = S!val!0,
 t = S!val!1,
 d = S!val!1,
 b = S!val!0,
 g = [else -> S!val!2],
 f = [(S!val!2, S!val!0) -> S!val!4, else -> S!val!3]]
